## Library loading

In [52]:
import pandas as pd
import geopandas as gpd

## Year lookups

In [53]:
fy_lookup = pd.read_csv("../../data/Financial year lookup.csv")

fy_lookup['year_month'] = pd.to_datetime(fy_lookup['year_month'], format='%d/%m/%Y')

## Shapefile

In [54]:
# --- Parameters ---
shapefile_path = "../../data/Local_Authority_Districts_(May_2025)_Boundaries_UK_BFC_(V2)/Local_Authority_Districts_(May_2025)_Boundaries_UK_BFC_(V2).shp"  # Change to your shapefile path
id_column = "LAD25CD"       # Column where first letter is E/W/S/N
name_column = "LAD25NM"   # Column with local authority names

# --- Load shapefile ---
gdf = gpd.read_file(shapefile_path)

# Filter to only England
gdf = gdf[gdf[id_column].str[0].isin(["E"])]

# Ensure consistent projection
gdf = gdf.to_crs(epsg=27700)  # British National Grid

## Function to update LA codes and names - default for dwelling stock

In [55]:
## Function to replace LA code and name pair to fix LA changes
def replace_la_pair(df, old_code, old_area, new_code, new_area, code_col='NewONScode', area_col='Area'):
    """Replace NewONScode and Area where both match old values.
    Prints number of rows changed and returns the dataframe."""
    mask = (df[code_col] == old_code) & (df[area_col] == old_area)
    changed = int(mask.sum())
    if changed:
        df.loc[mask, [code_col, area_col]] = [new_code, new_area]
    print(f"Updated rows ({old_code}, {old_area} -> {new_code}, {new_area}): {changed}")
    return df

## Population data loading

In [56]:
pop_by_la = pd.read_excel('../../data/population estimates.xlsx', sheet_name='Data', skiprows=6, na_values=['-'])

# detect columns that look like year (e.g."1997")
year_cols = pop_by_la.filter(regex=r'(^\d{4}$)').columns

id_vars = [c for c in pop_by_la.columns if c not in year_cols]

pop_by_la = pd.melt(
    pop_by_la,
    id_vars=id_vars,
    value_vars=year_cols,
    var_name='year',
    value_name='population'
)

# year to int
pop_by_la['year'] = pop_by_la['year'].astype(int)   
              
# filter to keep only England LAs
pop_by_la = pop_by_la[pop_by_la['LAD25CD'].str[0].isin(["E"])]

pop_by_la = pd.merge(
    pop_by_la,
    fy_lookup[['year_month','mid year pop estimate month']],
    how='left',
    left_on='year',
    right_on='mid year pop estimate month'
)


## Annual survey of earnings and hours

In [57]:
ashe_la = pd.read_excel('../../data/earnings.xlsx', sheet_name='Median earnings by LA', na_values=['-'])

# detect columns that look like year (e.g."1997")
year_cols = ashe_la.filter(regex=r'(^\d{4}$)').columns

id_vars = [c for c in ashe_la.columns if c not in year_cols]

ashe_la = pd.melt(
    ashe_la,
    id_vars=id_vars,
    value_vars=year_cols,
    var_name='year',
    value_name='ashe_weekly'
)

# year to int
ashe_la['year'] = ashe_la['year'].astype(int)   
ashe_la['financial_year'] = ashe_la['year'].astype(int).apply(lambda y: f"{y-1}/{str(y)[-2:]}")

ashe_la.drop(columns=['year'], inplace=True)
              
ashe_la = pd.merge(
    ashe_la,
    fy_lookup[['year_month','Financial year']],
    how='left',
    left_on='financial_year',
    right_on='Financial year'
)

ashe_la.drop(columns=['financial_year','Financial year'], inplace=True)

ashe_la.rename(columns={'Code': 'LAD25CD', 'Description': 'Description_ashe'}, inplace=True)

In [58]:
ashe_la

,Description_ashe,LAD25CD,ashe_weekly,year_month
0,Adur,E07000223,327.2,2007-04-01
1,Adur,E07000223,327.2,2007-05-01
2,Adur,E07000223,327.2,2007-06-01
3,Adur,E07000223,327.2,2007-07-01
4,Adur,E07000223,327.2,2007-08-01
...,...,...,...,...
63931,York UA,E06000014,619.2,2024-11-01
63932,York UA,E06000014,619.2,2024-12-01
63933,York UA,E06000014,619.2,2025-01-01
63934,York UA,E06000014,619.2,2025-02-01


## Base rate

In [59]:
base_rate = pd.read_excel('../../data/base rate.xlsx', sheet_name='Sheet1')

## Claimant count

In [60]:
claimant_count = pd.read_excel('../../data/claimant count.xlsx', sheet_name='claimant count (proportion)', skiprows=7, skipfooter=40)  

claimant_count = replace_la_pair(claimant_count, 'E08000016', 'Barnsley', 'E08000038', 'Barnsley', code_col='mnemonic', area_col='local authority: district / unitary (as of April 2023)')
claimant_count = replace_la_pair(claimant_count, 'E08000019', 'Sheffield', 'E08000039', 'Sheffield', code_col='mnemonic', area_col='local authority: district / unitary (as of April 2023)')

year_cols = claimant_count.filter(regex=r'(^\d{4}-\d{2}-\d{2} 00:00:00$)').columns

id_vars = [c for c in claimant_count.columns if c not in year_cols]

claimant_count = pd.melt(
    claimant_count,
    id_vars=id_vars,
    value_vars=year_cols,
    var_name='year_month',
    value_name='claimant_count_prop'
)
              
# filter to keep only England LAs
claimant_count = claimant_count[claimant_count['mnemonic'].str[0].isin(["E"])]

claimant_count['year_month'] = pd.to_datetime(claimant_count['year_month']).dt.date
claimant_count['year_month'] = pd.to_datetime(claimant_count['year_month'], format='%Y-%m-%d')

Updated rows (E08000016, Barnsley -> E08000038, Barnsley): 1
Updated rows (E08000019, Sheffield -> E08000039, Sheffield): 1


## Planning applications

In [61]:
planning_apps = pd.read_csv('../../data/planning data (1).csv', skiprows=2, na_values=['..'], usecols=range(7))

planning_apps = planning_apps.rename(columns={'LPACD': 'LAD25CD_p', 'LPANM': 'LAD25NM_p'})

## Durham to County Durham UA 09 update
planning_apps = replace_la_pair(planning_apps, 'A1340', 'Wear Valley', 'E06000047', 'County Durham', code_col='LAD25CD_p', area_col='LAD25NM_p')
planning_apps = replace_la_pair(planning_apps, 'G1305', 'Chester-le-Street', 'E06000047', 'County Durham', code_col='LAD25CD_p', area_col='LAD25NM_p')
planning_apps = replace_la_pair(planning_apps, 'H1325', 'Easington', 'E06000047', 'County Durham', code_col='LAD25CD_p', area_col='LAD25NM_p')
planning_apps = replace_la_pair(planning_apps, 'M1330', 'Sedgefield', 'E06000047', 'County Durham', code_col='LAD25CD_p', area_col='LAD25NM_p')
planning_apps = replace_la_pair(planning_apps, 'V1315', 'Derwentside', 'E06000047', 'County Durham', code_col='LAD25CD_p', area_col='LAD25NM_p')
planning_apps = replace_la_pair(planning_apps, 'W1335', 'Teesdale', 'E06000047', 'County Durham', code_col='LAD25CD_p', area_col='LAD25NM_p')
planning_apps = replace_la_pair(planning_apps, 'Z1320', 'Durham', 'E06000047', 'County Durham', code_col='LAD25CD_p', area_col='LAD25NM_p')
	
# Cheshire East update 09 update
planning_apps = replace_la_pair(planning_apps, 'B0610', 'Congleton', 'E06000049', 'Cheshire East', code_col='LAD25CD_p', area_col='LAD25NM_p')
planning_apps = replace_la_pair(planning_apps, 'K0615', 'Crewe and Nantwich', 'E06000049', 'Cheshire East', code_col='LAD25CD_p', area_col='LAD25NM_p')
planning_apps = replace_la_pair(planning_apps, 'T0633', 'Macclesfield', 'E06000049', 'Cheshire East', code_col='LAD25CD_p', area_col='LAD25NM_p')

# Cheshire West update	09 update
planning_apps = replace_la_pair(planning_apps, 'X0605', 'Chester', 'E06000050', 'Cheshire West and Chester', code_col='LAD25CD_p', area_col='LAD25NM_p')
planning_apps = replace_la_pair(planning_apps, 'P0620', 'Ellesmere Port and Neston', 'E06000050', 'Cheshire West and Chester', code_col='LAD25CD_p', area_col='LAD25NM_p')
planning_apps = replace_la_pair(planning_apps, 'L0635', 'Vale Royal', 'E06000050', 'Cheshire West and Chester', code_col='LAD25CD_p', area_col='LAD25NM_p')

# Shropshire UA update 09 update
planning_apps = replace_la_pair(planning_apps, 'B3220', 'Shrewsbury and Atcham', 'E06000051', 'Shropshire', code_col='LAD25CD_p', area_col='LAD25NM_p')
planning_apps = replace_la_pair(planning_apps, 'J3205', 'Bridgnorth', 'E06000051', 'Shropshire', code_col='LAD25CD_p', area_col='LAD25NM_p')
planning_apps = replace_la_pair(planning_apps, 'K3225', 'South Shropshire', 'E06000051', 'Shropshire', code_col='LAD25CD_p', area_col='LAD25NM_p')
planning_apps = replace_la_pair(planning_apps, 'N3210', 'North Shropshire', 'E06000051', 'Shropshire', code_col='LAD25CD_p', area_col='LAD25NM_p')
planning_apps = replace_la_pair(planning_apps, 'X3215', 'Oswestry', 'E06000051', 'Shropshire', code_col='LAD25CD_p', area_col='LAD25NM_p')

# Cornwall UA update 09 update
planning_apps = replace_la_pair(planning_apps, 'C0820', 'North Cornwall', 'E06000052', 'Cornwall', code_col='LAD25CD_p', area_col='LAD25NM_p')
planning_apps = replace_la_pair(planning_apps, 'K0805', 'Caradon', 'E06000052', 'Cornwall', code_col='LAD25CD_p', area_col='LAD25NM_p')
planning_apps = replace_la_pair(planning_apps, 'L0825', 'Penwith', 'E06000052', 'Cornwall', code_col='LAD25CD_p', area_col='LAD25NM_p')
planning_apps = replace_la_pair(planning_apps, 'P0810', 'Carrick', 'E06000052', 'Cornwall', code_col='LAD25CD_p', area_col='LAD25NM_p')
planning_apps = replace_la_pair(planning_apps, 'Q0830', 'Restormel', 'E06000052', 'Cornwall', code_col='LAD25CD_p', area_col='LAD25NM_p')
planning_apps = replace_la_pair(planning_apps, 'Y0815', 'Kerrier', 'E06000052', 'Cornwall', code_col='LAD25CD_p', area_col='LAD25NM_p')

# Wiltshire UA update 09 update
planning_apps = replace_la_pair(planning_apps, 'E3905', 'Kennet', 'E06000054', 'Wiltshire', code_col='LAD25CD_p', area_col='LAD25NM_p')
planning_apps = replace_la_pair(planning_apps, 'F3925', 'West Wiltshire', 'E06000054', 'Wiltshire', code_col='LAD25CD_p', area_col='LAD25NM_p')
planning_apps = replace_la_pair(planning_apps, 'J3910', 'North Wiltshire', 'E06000054', 'Wiltshire', code_col='LAD25CD_p', area_col='LAD25NM_p')
planning_apps = replace_la_pair(planning_apps, 'T3915', 'Salisbury', 'E06000054', 'Wiltshire', code_col='LAD25CD_p', area_col='LAD25NM_p')

# Central Bedfordshire UA update 09 update
planning_apps = replace_la_pair(planning_apps, 'J0215', 'Mid Bedfordshire', 'E06000056', 'Central Bedfordshire', code_col='LAD25CD_p', area_col='LAD25NM_p')
planning_apps = replace_la_pair(planning_apps, 'N0220', 'South Bedfordshire', 'E06000056', 'Central Bedfordshire', code_col='LAD25CD_p', area_col='LAD25NM_p')

# Northumberland UA update 09 update
planning_apps = replace_la_pair(planning_apps, 'F2930', 'Wansbeck', 'E06000057', 'Northumberland', code_col='LAD25CD_p', area_col='LAD25NM_p')
planning_apps = replace_la_pair(planning_apps, 'N2915', 'Blyth Valley', 'E06000057', 'Northumberland', code_col='LAD25CD_p', area_col='LAD25NM_p')
planning_apps = replace_la_pair(planning_apps, 'Q2908', 'Alnwick', 'E06000057', 'Northumberland', code_col='LAD25CD_p', area_col='LAD25NM_p')
planning_apps = replace_la_pair(planning_apps, 'R2928', 'Tynedale', 'E06000057', 'Northumberland', code_col='LAD25CD_p', area_col='LAD25NM_p')
planning_apps = replace_la_pair(planning_apps, 'T2920', 'Castle Morpeth', 'E06000057', 'Northumberland', code_col='LAD25CD_p', area_col='LAD25NM_p')
planning_apps = replace_la_pair(planning_apps, 'V2913', 'Berwick-upon-Tweed', 'E06000057', 'Northumberland', code_col='LAD25CD_p', area_col='LAD25NM_p')
	
# Bournemouth, christchurch and poole UA update '19 update
planning_apps = replace_la_pair(planning_apps, 'E06000028', 'Bournemouth', 'E06000058', 'Bournemouth, Christchurch and Poole', code_col='LAD25CD_p', area_col='LAD25NM_p')
planning_apps = replace_la_pair(planning_apps, 'E06000029', 'Poole', 'E06000058', 'Bournemouth, Christchurch and Poole', code_col='LAD25CD_p', area_col='LAD25NM_p')
planning_apps = replace_la_pair(planning_apps, 'E07000048', 'Christchurch', 'E06000058', 'Bournemouth, Christchurch and Poole', code_col='LAD25CD_p', area_col='LAD25NM_p')

# Dorset UA update '19 update
planning_apps = replace_la_pair(planning_apps, 'E07000049', 'East Dorset', 'E06000059', 'Dorset', code_col='LAD25CD_p', area_col='LAD25NM_p')
planning_apps = replace_la_pair(planning_apps, 'E07000050', 'North Dorset', 'E06000059', 'Dorset', code_col='LAD25CD_p', area_col='LAD25NM_p')
planning_apps = replace_la_pair(planning_apps, 'E07000051', 'Purbeck', 'E06000059', 'Dorset', code_col='LAD25CD_p', area_col='LAD25NM_p')
planning_apps = replace_la_pair(planning_apps, 'E07000052', 'West Dorset', 'E06000059', 'Dorset', code_col='LAD25CD_p', area_col='LAD25NM_p')
planning_apps = replace_la_pair(planning_apps, 'E07000053', 'Weymouth and Portland', 'E06000059', 'Dorset', code_col='LAD25CD_p', area_col='LAD25NM_p')

# Buckinghamshire UA update '19 update
planning_apps = replace_la_pair(planning_apps, 'E07000004', 'Aylesbury Vale', 'E06000060', 'Buckinghamshire', code_col='LAD25CD_p', area_col='LAD25NM_p')
planning_apps = replace_la_pair(planning_apps, 'E07000005', 'Chiltern', 'E06000060', 'Buckinghamshire', code_col='LAD25CD_p', area_col='LAD25NM_p')
planning_apps = replace_la_pair(planning_apps, 'E07000006', 'South Bucks', 'E06000060', 'Buckinghamshire', code_col='LAD25CD_p', area_col='LAD25NM_p')
planning_apps = replace_la_pair(planning_apps, 'E07000007', 'Wycombe', 'E06000060', 'Buckinghamshire', code_col='LAD25CD_p', area_col='LAD25NM_p')

# East Suffolk update '19 update
planning_apps = replace_la_pair(planning_apps, 'E07000205', 'Suffolk Coastal', 'E07000244', 'East Suffolk', code_col='LAD25CD_p', area_col='LAD25NM_p')
planning_apps = replace_la_pair(planning_apps, 'E07000206', 'Waveney', 'E07000244', 'East Suffolk', code_col='LAD25CD_p', area_col='LAD25NM_p')

# West Suffolk update '19 update
planning_apps = replace_la_pair(planning_apps, 'E07000201', 'Forest Heath', 'E07000245', 'West Suffolk', code_col='LAD25CD_p', area_col='LAD25NM_p')
planning_apps = replace_la_pair(planning_apps, 'E07000204', 'St Edmundsbury', 'E07000245', 'West Suffolk', code_col='LAD25CD_p', area_col='LAD25NM_p')

# Folkestone and Hythe update '18 update
planning_apps = replace_la_pair(planning_apps, 'E07000112', 'Shepway', 'E07000112', 'Folkestone and Hythe', code_col='LAD25CD_p', area_col='LAD25NM_p')

# North Northamptonshire update '21 update
planning_apps = replace_la_pair(planning_apps, 'E07000150', 'Corby', 'E06000061', 'North Northamptonshire', code_col='LAD25CD_p', area_col='LAD25NM_p')
planning_apps = replace_la_pair(planning_apps, 'E07000152', 'East Northamptonshire', 'E06000061', 'North Northamptonshire', code_col='LAD25CD_p', area_col='LAD25NM_p')
planning_apps = replace_la_pair(planning_apps, 'E07000153', 'Kettering', 'E06000061', 'North Northamptonshire', code_col='LAD25CD_p', area_col='LAD25NM_p')
planning_apps = replace_la_pair(planning_apps, 'E07000156', 'Wellingborough', 'E06000061', 'North Northamptonshire', code_col='LAD25CD_p', area_col='LAD25NM_p')

# West Northamptonshire update '21 update
planning_apps = replace_la_pair(planning_apps, 'E07000151', 'Daventry', 'E06000062', 'West Northamptonshire', code_col='LAD25CD_p', area_col='LAD25NM_p')
planning_apps = replace_la_pair(planning_apps, 'E07000154', 'Northampton', 'E06000062', 'West Northamptonshire', code_col='LAD25CD_p', area_col='LAD25NM_p')
planning_apps = replace_la_pair(planning_apps, 'E07000155', 'South Northamptonshire', 'E06000062', 'West Northamptonshire', code_col='LAD25CD_p', area_col='LAD25NM_p')

# Cumberland update '23 update
planning_apps = replace_la_pair(planning_apps, 'E07000026', 'Allerdale', 'E06000063', 'Cumberland', code_col='LAD25CD_p', area_col='LAD25NM_p')
planning_apps = replace_la_pair(planning_apps, 'E07000028', 'Carlisle', 'E06000063', 'Cumberland', code_col='LAD25CD_p', area_col='LAD25NM_p')
planning_apps = replace_la_pair(planning_apps, 'E07000029', 'Copeland', 'E06000063', 'Cumberland', code_col='LAD25CD_p', area_col='LAD25NM_p')

# Westmorland and Furness update '23 update
planning_apps = replace_la_pair(planning_apps, 'E07000027', 'Barrow-in-Furness', 'E06000064', 'Westmorland and Furness', code_col='LAD25CD_p', area_col='LAD25NM_p')
planning_apps = replace_la_pair(planning_apps, 'E07000030', 'Eden', 'E06000064', 'Westmorland and Furness', code_col='LAD25CD_p', area_col='LAD25NM_p')
planning_apps = replace_la_pair(planning_apps, 'E07000031', 'South Lakeland', 'E06000064', 'Westmorland and Furness', code_col='LAD25CD_p', area_col='LAD25NM_p')

# North Yorkshire update '23 update
planning_apps = replace_la_pair(planning_apps, 'E07000163', 'Craven', 'E06000065', 'North Yorkshire', code_col='LAD25CD_p', area_col='LAD25NM_p')
planning_apps = replace_la_pair(planning_apps, 'E07000164', 'Hambleton', 'E06000065', 'North Yorkshire', code_col='LAD25CD_p', area_col='LAD25NM_p')
planning_apps = replace_la_pair(planning_apps, 'E07000165', 'Harrogate', 'E06000065', 'North Yorkshire', code_col='LAD25CD_p', area_col='LAD25NM_p')
planning_apps = replace_la_pair(planning_apps, 'E07000166', 'Richmondshire', 'E06000065', 'North Yorkshire', code_col='LAD25CD_p', area_col='LAD25NM_p')
planning_apps = replace_la_pair(planning_apps, 'E07000167', 'Ryedale', 'E06000065', 'North Yorkshire', code_col='LAD25CD_p', area_col='LAD25NM_p')
planning_apps = replace_la_pair(planning_apps, 'E07000168', 'Scarborough', 'E06000065', 'North Yorkshire', code_col='LAD25CD_p', area_col='LAD25NM_p')
planning_apps = replace_la_pair(planning_apps, 'E07000169', 'Selby', 'E06000065', 'North Yorkshire', code_col='LAD25CD_p', area_col='LAD25NM_p')

# Somerset update '23 update
planning_apps = replace_la_pair(planning_apps, 'E07000163', 'Craven', 'E06000066', 'Somerset', code_col='LAD25CD_p', area_col='LAD25NM_p')
planning_apps = replace_la_pair(planning_apps, 'E07000187', 'Mendip', 'E06000066', 'Somerset', code_col='LAD25CD_p', area_col='LAD25NM_p')
planning_apps = replace_la_pair(planning_apps, 'E07000188', 'Sedgemoor', 'E06000066', 'Somerset', code_col='LAD25CD_p', area_col='LAD25NM_p')
planning_apps = replace_la_pair(planning_apps, 'E07000189', 'South Somerset', 'E06000066', 'Somerset', code_col='LAD25CD_p', area_col='LAD25NM_p')
planning_apps = replace_la_pair(planning_apps, 'E07000246', 'Somerset West and Taunton', 'E06000066', 'Somerset', code_col='LAD25CD_p', area_col='LAD25NM_p')

#planning_apps = planning_apps.rename(columns={'LPACD': 'LAD25CD_p', 'LPANM': 'LAD25NM_p'})

planning_apps = planning_apps.groupby(
    ['Region', 'LAD25NM_p',	'LAD25CD_p', 'Quarter'],
    as_index=False
).agg({
    'Total decisions; grand total (all)': 'sum',
    'Total granted; grand total (all)': 'sum',	
    'Total refused; grand total (all)': 'sum'
}) 

planning_apps = pd.merge(
    planning_apps,
    fy_lookup[['year_month','Quarters']],
    how='left',
    left_on='Quarter',
    right_on='Quarters'
).drop(
     columns=['Region', 'Quarter', 'Quarters']
 ).rename(columns={'LAD25CD_p': 'LAD25CD', 'LAD25NM_p': 'LAD25NM'})

Updated rows (A1340, Wear Valley -> E06000047, County Durham): 33
Updated rows (G1305, Chester-le-Street -> E06000047, County Durham): 33
Updated rows (H1325, Easington -> E06000047, County Durham): 33
Updated rows (M1330, Sedgefield -> E06000047, County Durham): 33
Updated rows (V1315, Derwentside -> E06000047, County Durham): 33
Updated rows (W1335, Teesdale -> E06000047, County Durham): 33
Updated rows (Z1320, Durham -> E06000047, County Durham): 33
Updated rows (B0610, Congleton -> E06000049, Cheshire East): 33
Updated rows (K0615, Crewe and Nantwich -> E06000049, Cheshire East): 33
Updated rows (T0633, Macclesfield -> E06000049, Cheshire East): 33
Updated rows (X0605, Chester -> E06000050, Cheshire West and Chester): 33
Updated rows (P0620, Ellesmere Port and Neston -> E06000050, Cheshire West and Chester): 33
Updated rows (L0635, Vale Royal -> E06000050, Cheshire West and Chester): 33
Updated rows (B3220, Shrewsbury and Atcham -> E06000051, Shropshire): 33
Updated rows (J3205, Br

In [62]:
planning_apps

,LAD25NM,LAD25CD,Total decisions; grand total (all),Total granted; grand total (all),Total refused; grand total (all),year_month
0,Amber Valley,E07000032,262,232,15,2001-01-01
1,Amber Valley,E07000032,262,232,15,2001-02-01
2,Amber Valley,E07000032,262,232,15,2001-03-01
3,Amber Valley,E07000032,316,284,22,2001-04-01
4,Amber Valley,E07000032,316,284,22,2001-05-01
...,...,...,...,...,...,...
92968,York,E06000014,306,251,55,2025-02-01
92969,York,E06000014,306,251,55,2025-03-01
92970,York,E06000014,374,301,73,2025-04-01
92971,York,E06000014,374,301,73,2025-05-01


## Dwelling Stock

In [63]:
## Function to replace LA code and name pair to fix LA changes
def replace_la_pair(df, old_code, old_area, new_code, new_area, code_col='NewONScode', area_col='Area'):
    """Replace NewONScode and Area where both match old values.
    Prints number of rows changed and returns the dataframe."""
    mask = (df[code_col] == old_code) & (df[area_col] == old_area)
    changed = int(mask.sum())
    if changed:
        df.loc[mask, [code_col, area_col]] = [new_code, new_area]
    print(f"Updated rows ({old_code}, {old_area} -> {new_code}, {new_area}): {changed}")
    return df

In [64]:
dwelling_stock = pd.read_excel('../../data/dwelling stock.xlsx', sheet_name='LT_125_unrounded', skiprows=5, na_values=['[x]', '[z]'])

dwelling_stock = dwelling_stock.drop(columns=['OldONScode'])

# filter to keep only England LAs
dwelling_stock = dwelling_stock[dwelling_stock['NewONScode'].str[0].isin(["E"])]

year_cols = dwelling_stock.filter(regex=r'(^\d{4}$)').columns

id_vars = [c for c in dwelling_stock.columns if c not in year_cols]

dwelling_stock = pd.melt(
    dwelling_stock,
    id_vars=id_vars,
    value_vars=year_cols,
    var_name='year',
    value_name='dwelling_stock'
)

## Durham to County Durham UA 09 update
dwelling_stock = replace_la_pair(dwelling_stock, 'E10000010', 'Durham', 'E06000047', 'County Durham UA')

# Cheshire East update 09 update
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000014', 'Congleton', 'E06000049', 'Cheshire East UA')
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000015', 'Crewe and Nantwich', 'E06000049', 'Cheshire East UA')
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000017', 'Macclesfield', 'E06000049', 'Cheshire East UA')

# Cheshire West update	09 update
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000013', 'Chester', 'E06000050', 'Cheshire West and Chester UA')
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000016', 'Ellesmere Port and Neston', 'E06000050', 'Cheshire West and Chester UA')
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000018', 'Vale Royal', 'E06000050', 'Cheshire West and Chester UA')

# Shropshire UA update 09 update
dwelling_stock = replace_la_pair(dwelling_stock, 'E10000026', 'Shropshire', 'E06000051', 'Shropshire UA')

# Cornwall UA update 09 update
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000019', 'Caradon', 'E06000052', 'Cornwall UA')
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000020', 'Carrick', 'E06000052', 'Cornwall UA')
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000021', 'Kerrier', 'E06000052', 'Cornwall UA')
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000022', 'North Cornwall', 'E06000052', 'Cornwall UA')
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000023', 'Penwith', 'E06000052', 'Cornwall UA')
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000024', 'Restormel', 'E06000052', 'Cornwall UA')

# Wiltshire UA update 09 update
dwelling_stock = replace_la_pair(dwelling_stock, 'E10000033', 'Wiltshire', 'E06000054', 'Wiltshire UA')

# Bedford UA update 09 update
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000002', 'Bedford', 'E06000055', 'Bedford UA')

# Central Bedfordshire UA update 09 update
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000001', 'Mid Bedfordshire', 'E06000056', 'Central Bedfordshire UA')
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000003', 'South Bedfordshire', 'E06000056', 'Central Bedfordshire UA')

# Northumberland UA update 09 update
dwelling_stock = replace_la_pair(dwelling_stock, 'E10000022', 'Northumberland', 'E06000057', 'Northumberland UA')
	
# Bournemouth, christchurch and poole UA update '19 update
dwelling_stock = replace_la_pair(dwelling_stock, 'E06000028', 'Bournemouth UA', 'E06000058', 'Bournemouth, Christchurch and Poole UA')
dwelling_stock = replace_la_pair(dwelling_stock, 'E06000029', 'Poole UA', 'E06000058', 'Bournemouth, Christchurch and Poole UA')
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000048', 'Christchurch', 'E06000058', 'Bournemouth, Christchurch and Poole UA')

# Dorset UA update '19 update
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000049', 'East Dorset', 'E06000059', 'Dorset UA')
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000050', 'North Dorset', 'E06000059', 'Dorset UA')
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000051', 'Purbeck', 'E06000059', 'Dorset UA')
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000052', 'West Dorset', 'E06000059', 'Dorset UA')
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000053', 'Weymouth and Portland', 'E06000059', 'Dorset UA')

# Buckinghamshire UA update '19 update
dwelling_stock = replace_la_pair(dwelling_stock, 'E10000002', 'Buckinghamshire', 'E06000060', 'Buckinghamshire UA')

# East Suffolk update '19 update
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000205', 'Suffolk Coastal', 'E07000244', 'East Suffolk')
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000206', 'Waveney', 'E07000244', 'East Suffolk')

# West Suffolk update '19 update
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000201', 'Forest Heath', 'E07000245', 'West Suffolk')
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000204', 'St. Edmundsbury', 'E07000245', 'West Suffolk')

# North Northamptonshire update '21 update
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000150', 'Corby', 'E06000061', 'North Northamptonshire')
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000152', 'East Northamptonshire', 'E06000061', 'North Northamptonshire')
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000153', 'Kettering', 'E06000061', 'North Northamptonshire')
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000156', 'Wellingborough', 'E06000061', 'North Northamptonshire')

# West Northamptonshire update '21 update
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000151', 'Daventry', 'E06000062', 'West Northamptonshire')
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000154', 'Northampton', 'E06000062', 'West Northamptonshire')
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000155', 'South Northamptonshire', 'E06000062', 'West Northamptonshire')

# Cumberland update '23 update
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000026', 'Allerdale', 'E06000063', 'Cumberland')
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000028', 'Carlisle', 'E06000063', 'Cumberland')
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000029', 'Copeland', 'E06000063', 'Cumberland')

# Westmorland and Furness update '23 update
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000027', 'Barrow-in-Furness', 'E06000064', 'Westmorland and Furness')
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000030', 'Eden', 'E06000064', 'Westmorland and Furness')
dwelling_stock = replace_la_pair(dwelling_stock, 'E07000031', 'South Lakeland', 'E06000064', 'Westmorland and Furness')

# North Yorkshire update '23 update
dwelling_stock = replace_la_pair(dwelling_stock, 'E10000023', 'North Yorkshire', 'E06000065', 'North Yorkshire')

# Somerset update '23 update
dwelling_stock = replace_la_pair(dwelling_stock, 'E10000027', 'Somerset', 'E06000066', 'Somerset')

# Sheffield and Barnsley ons recode in dwelling stock
dwelling_stock = replace_la_pair(dwelling_stock, 'E08000016', 'Barnsley', 'E08000038', 'Barnsley')
dwelling_stock = replace_la_pair(dwelling_stock, 'E08000019', 'Sheffield', 'E08000039', 'Sheffield')


# year to int
dwelling_stock['year'] = dwelling_stock['year'].astype(int)   

# Summarise dwelling_stock so each (NewONScode, Area, year) is unique
dwelling_stock = dwelling_stock.groupby(
    ['NewONScode', 'Area', 'year'],
    as_index=False
).agg({'dwelling_stock': 'sum'})  # use 'mean' or other agg if appropriate

dwelling_stock = pd.merge(
    dwelling_stock,
    fy_lookup[['year_month','dwelling_stock_year']],
    how='left',
    left_on='year',
    right_on='dwelling_stock_year'
).drop(columns = ['dwelling_stock_year'])



Updated rows (E10000010, Durham -> E06000047, County Durham UA): 24
Updated rows (E07000014, Congleton -> E06000049, Cheshire East UA): 24
Updated rows (E07000015, Crewe and Nantwich -> E06000049, Cheshire East UA): 24
Updated rows (E07000017, Macclesfield -> E06000049, Cheshire East UA): 24
Updated rows (E07000013, Chester -> E06000050, Cheshire West and Chester UA): 24
Updated rows (E07000016, Ellesmere Port and Neston -> E06000050, Cheshire West and Chester UA): 24
Updated rows (E07000018, Vale Royal -> E06000050, Cheshire West and Chester UA): 24
Updated rows (E10000026, Shropshire -> E06000051, Shropshire UA): 24
Updated rows (E07000019, Caradon -> E06000052, Cornwall UA): 24
Updated rows (E07000020, Carrick -> E06000052, Cornwall UA): 24
Updated rows (E07000021, Kerrier -> E06000052, Cornwall UA): 24
Updated rows (E07000022, North Cornwall -> E06000052, Cornwall UA): 24
Updated rows (E07000023, Penwith -> E06000052, Cornwall UA): 24
Updated rows (E07000024, Restormel -> E06000052

## ORR Station entry and exit numbers  by station and LA

2003/04 data had to be interpolated. This was done by taking the midpoint between 2002/03 and 2004/05. This was done directly in the Excel file.

I've also had to add the ONS LA codes for ease when joining the data. This was also done directly in the Excel file.

In [65]:
raw_orr_data = pd.read_excel("../../data/station entries and exits.xlsx", sheet_name="1415a_Entries_and_Exits", skiprows = 3, na_values = ['[z]', '[x]'])

In [66]:
raw_orr_data = raw_orr_data[~raw_orr_data['Region'].isin(["Wales", "Scotland", "[z]"])& ~raw_orr_data['Region'].isna()]



In [67]:
raw_orr_data = pd.melt(
    raw_orr_data,
    id_vars=[col for col in raw_orr_data.columns if not pd.Series(col).str.match(r'^\d{4}/\d{2}$')[0]],   # columns to keep fixed
    value_vars=raw_orr_data.filter(regex=r'^\d{4}/\d{2}$').columns,    # columns to unpivot (optional)
    var_name='Financial year',                    # name for new variable column
    value_name='rail_station_entry_exit'                      # name for new value column
)

In [68]:
orr_data = raw_orr_data.groupby(
    ['Financial year', 'Local authority: district or unitary', 'Local authority code'],
      as_index=False
)['rail_station_entry_exit'].sum()

In [69]:
orr_data = pd.merge(
    orr_data,
    fy_lookup[['year_month','Financial year']],
    how='left',
    left_on='Financial year',
    right_on='Financial year'
)

In [70]:
orr_data.rename(columns={'Local authority code': 'LAD25CD', 'Local authority: district or unitary': 'LAD25NM'}, inplace=True)

orr_data.drop(columns=['Financial year'], inplace=True)

In [71]:
orr_data

,LAD25NM,LAD25CD,rail_station_entry_exit,year_month
0,Adur,E07000223,1797095.0,1997-04-01
1,Adur,E07000223,1797095.0,1997-05-01
2,Adur,E07000223,1797095.0,1997-06-01
3,Adur,E07000223,1797095.0,1997-07-01
4,Adur,E07000223,1797095.0,1997-08-01
...,...,...,...,...
93955,York,E06000014,9274308.0,2023-11-01
93956,York,E06000014,9274308.0,2023-12-01
93957,York,E06000014,9274308.0,2024-01-01
93958,York,E06000014,9274308.0,2024-02-01


## GDP data

In [72]:
gdp_data = pd.read_excel("../../data/monthlygdpto4dp.xlsx", sheet_name="Data_table", skiprows=3)

In [73]:
gdp_data['Month'] = pd.to_datetime(gdp_data['Month'], format='%Y%b')

In [74]:
gdp_data = gdp_data[['Month', 'Monthly GDP (A-T)']]
gdp_data.rename(columns={'Monthly GDP (A-T)': 'GDP'}, inplace=True)

In [75]:
gdp_orr_data = pd.merge(orr_data, gdp_data, how='left', left_on='year_month', right_on='Month')

## CPIH

In [76]:
cpih = pd.read_csv("../../data/cpih.csv", skiprows=189)
cpih.rename(columns={'2025 Q2': 'year_month', '4.1': 'CPIH'}, inplace=True)
cpih['year_month'] = pd.to_datetime(cpih['year_month'], format='%Y %b')


## Combine data

In [ ]:
# dwelling
additional_data = pd.merge(
    gdf[['LAD25CD', 'LAD25NM']],
    dwelling_stock,
    how='left',
    left_on='LAD25CD',
    right_on='NewONScode'
).drop(columns=['NewONScode', 'Area', 'year'])

# population
additional_data = pd.merge(
    additional_data,
    pop_by_la,
    how='left',
    left_on=['LAD25CD', 'year_month'],
    right_on=['LAD25CD', 'year_month']
).drop(columns=['year', 'mid year pop estimate month', 'local authority: district / unitary (as of April 2023)'])

# earnings
additional_data = pd.merge(
    additional_data,
    ashe_la,
    how='left',
    left_on=['LAD25CD', 'year_month'],
    right_on=['LAD25CD', 'year_month']
)

# base rate
additional_data = pd.merge(
    additional_data,
    base_rate,
    how='left',
    left_on='year_month',
    right_on='Date' 
).drop(columns=['Date'])


# claimant count
additional_data = pd.merge(
    additional_data,
    claimant_count,
    how='left',
    left_on=['LAD25CD', 'year_month'],
    right_on=['mnemonic', 'year_month']
).drop(columns=['mnemonic', 'Description_ashe', 'local authority: district / unitary (as of April 2023)'])

# planning applications
additional_data = pd.merge(
    additional_data,
    planning_apps,
    how='left',
    left_on=['LAD25CD', 'year_month'],
    right_on=['LAD25CD', 'year_month']
)

# ORR data
additional_data = pd.merge(
    additional_data,
    orr_data,
    how='left',
    left_on=['LAD25CD', 'year_month'],
    right_on=['LAD25CD', 'year_month']
)

additional_data['rail_station_entry_exit'] = additional_data['rail_station_entry_exit'].fillna(0)

# GDP data
additional_data = pd.merge(
    additional_data,
    gdp_data,
    how='left',
    left_on='year_month',
    right_on='Month'
).drop(columns=['Month'])

# CPIH data
additional_data = pd.merge(
    additional_data,
    cpih,
    how='left',
    left_on='year_month',
    right_on='year_month'
)

additional_data.drop(columns=['LAD25NM_y', 'LAD25NM'], inplace=True).rename(columns={'LAD25NM_x': 'LAD25NM'}, inplace=True)

,LAD25CD,LAD25NM_x,dwelling_stock,year_month,population,ashe_weekly,base_rate,claimant_count_prop,LAD25NM_y,Total decisions; grand total (all),Total granted; grand total (all),Total refused; grand total (all),LAD25NM,rail_station_entry_exit,GDP,CPIH
0,E06000001,Hartlepool,41113.8,2007-04-01,90781.0,316.6,5.25,4.4,Hartlepool,199,190,8,Hartlepool,422020.0,83.4645,2.7
1,E06000001,Hartlepool,41113.8,2007-05-01,90781.0,316.6,5.50,4.3,Hartlepool,199,190,8,Hartlepool,422020.0,83.9159,2.5
2,E06000001,Hartlepool,41113.8,2007-06-01,90781.0,316.6,5.50,4.1,Hartlepool,199,190,8,Hartlepool,422020.0,84.0324,2.5
3,E06000001,Hartlepool,41113.8,2007-07-01,90969.0,316.6,5.75,4.1,Hartlepool,208,199,9,Hartlepool,422020.0,83.9629,2.0
4,E06000001,Hartlepool,41113.8,2007-08-01,90969.0,316.6,5.75,4,Hartlepool,208,199,9,Hartlepool,422020.0,84.4972,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63931,E09000033,Westminster,132895.0,2024-11-01,209996.0,947.4,4.75,4.2,Westminster,1245,1098,147,NaN,NaN,101.4706,3.5
63932,E09000033,Westminster,132895.0,2024-12-01,209996.0,947.4,4.75,4.2,Westminster,1245,1098,147,NaN,NaN,101.9167,3.5
63933,E09000033,Westminster,132895.0,2025-01-01,209996.0,947.4,4.75,4.2,Westminster,1236,1084,152,NaN,NaN,101.8995,3.9
63934,E09000033,Westminster,132895.0,2025-02-01,209996.0,947.4,4.50,4.4,Westminster,1236,1084,152,NaN,NaN,102.3612,3.7


In [ ]:
additional_data.to_excel("additional_data.xlsx", sheet_name='Sheet1', index=False, engine = 'openpyxl')

63936 rows expected